# Week 4 Lab 3 — POD-DeepONet for the Ghia cavity

<!-- FLOWMLLAB_COLAB_LAUNCH_V1 -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ehsan-Roohi/FlowMLLab/blob/main/notebooks/week04/W4_Lab3_DeepONet_Cavity_Student.ipynb)

<!-- MIE690A article-aligned validation v3 -->

This lab turns the validated lid-driven-cavity family into a compact **parameter-to-field neural operator**. Separate velocity and pressure POD trunks preserve field scaling, while two small branch heads receive Reynolds number. You will verify held-out velocity and direct-pressure accuracy, incompressibility, the pressure gauge, Ghia centerlines, seed sensitivity, and measured inference speed.

### Learning outcomes

By the end, you should be able to:

1. distinguish coordinate regression from branch–trunk operator factorization;
2. build a POD trunk without using the public held-out regression fields;
3. select rank and branch width using a complete validation case;
4. refit with three predeclared seeds and assess the public held-out regression cases under a frozen manifest;
5. compare the neural field with CFD and Ghia centerlines; and
6. explain why fast inference is a valid advantage even when the CFD labels remain the physical reference.

## Assessment contract: public held-out cases are not a new blind experiment
The course has already displayed Re=175, 275 and 375 in earlier lessons, and
some other project test cases have appeared as development/benchmark cases.
Keeping a case out of fitting is necessary, but does not make it unseen to a
student who has inspected it before. Here **public held-out regression test**
is the correct label for reused cases, including older tables labelled "blind".
Those historical labels describe the original author protocol, not a new claim
of student blindness. Do not revise settings using these results and call the
same assessment untouched.

Before final assessment, record a dated manifest containing development cases,
validation case, all candidates/seeds, thresholds, source/data hashes, and an
exposure log (case, earlier lesson, whether viewed). If an instructor provides
new cases, their fields remain withheld until that manifest is frozen. Otherwise
report a reproducible public-case study, **not a new blind generalization study**.
No synthetic replacement test or new CFD qualification is implied by this revision.

## Concept map

$$
\text{validated CFD snapshots}
\rightarrow \text{field-specific POD trunks}
\rightarrow \text{branch heads }b_k(Re)
\rightarrow (\widehat{\mathbf u},\widehat p)(Re,x,y)
\rightarrow \text{public held-out regression fields + Ghia + timing}.
$$

The model is

$$
\widehat{\mathbf u}=\overline{\mathbf u}+\sum_{k=1}^{r_u}b_k^{u}(Re)t_k^{u},
\qquad
\widehat p=\overline p+\sum_{k=1}^{r_p}b_k^{p}(\log Re)t_k^{p}.
$$

This is a restricted scalar-branch **POD-DeepONet**. It does not claim to learn an arbitrary function-to-function map; a true boundary-function operator is proposed in the exercises.

## Reproducibility contract

- Development cases: `Re = [100, 150, 200, 225, 250, 300, 350, 400]`
- Architecture-selection validation case: complete `Re = 225` field
- Public held-out regression cases: `Re = [175, 275, 375]`
- Velocity candidates: ranks `[3,4]` and widths `(16,16)`, `(32,32)`, `(64,64)`
- Pressure candidates: ranks `[2,3,4]`, widths `(4,)`, `(8,)`, `(8,8)`, `(16,16)`, `(32,32)`, and declared `Re`/`log(Re)` transforms
- Predeclared seeds: `690, 691, 692`
- Metrics: velocity-vector and zero-mean pressure relative $L_2$, plus wall, divergence, and gauge checks

Freeze the candidate set before assessing the public held-out regression metrics. These cases have already been exposed in the course; freezing a new manifest does not undo that exposure. Any redesign informed by these results is development, and a new blind claim requires newly withheld cases.

In [ ]:
# FLOWMLLAB_COLAB_BOOTSTRAP_V1
from pathlib import Path as _FlowMLLabPath
import os as _flowmllab_os
import subprocess as _flowmllab_subprocess
import sys as _flowmllab_sys

if "google.colab" in _flowmllab_sys.modules or _flowmllab_os.environ.get("COLAB_RELEASE_TAG"):
    _flowmllab_root = _FlowMLLabPath("/content/FlowMLLab")
    if not (_flowmllab_root / ".git").is_dir():
        _flowmllab_subprocess.run(
            [
                "git", "clone", "--depth", "1",
                "https://github.com/Ehsan-Roohi/FlowMLLab.git", str(_flowmllab_root),
            ],
            check=True,
        )
    _flowmllab_subprocess.run(
        [
            _flowmllab_sys.executable, "-m", "pip", "install", "-q", "-e",
            f"{_flowmllab_root}[test]",
        ],
        check=True,
    )
    _flowmllab_notebook_dir = _flowmllab_root / "notebooks/week04"
    _flowmllab_os.chdir(_flowmllab_notebook_dir)
    for _flowmllab_path in (_flowmllab_root, _flowmllab_notebook_dir):
        if str(_flowmllab_path) not in _flowmllab_sys.path:
            _flowmllab_sys.path.insert(0, str(_flowmllab_path))
    print("FlowMLLab ready:", _flowmllab_root)

from pathlib import Path
import sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Works from the repository root or from notebooks/week04.
ROOT = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "data" / "cavity_data.npz").exists()
)
sys.path.insert(0, str(ROOT / "common"))
import run_pod_deeponet_validation as poddon
import w4utils

RESULTS = ROOT / "results" / "pod_deeponet"
data = poddon.load_data()
print("course root:", ROOT)
print("dataset version:", str(data["package_version"]))
print("development Re:", data["Re"][data["split"] == "train"])
print("blind Re:", data["Re"][data["split"] == "test"])
assert np.all(data["accepted"]), "One or more CFD cases failed quality control"


## Before fitting: the branch–trunk–target data contract

A valid array shape does not establish a valid physical pairing. Each row must join **one physical case**, **its query coordinate**, and **the target at that coordinate**. Preserve case IDs and coordinates from the solver export before reshaping; do not reconstruct the reference metadata from the potentially wrong query array.

| Representation | Case input | Spatial representation | Supported claim |
|---|---|---|---|
| Coordinate MLP | Parameters concatenated with coordinates | Learned jointly | Parameter-to-field regression |
| This POD–DeepONet | Scalar Re (or log Re) | Fixed, training-derived POD modes | Restricted parametric field prediction |
| Sensor-based DeepONet | Function values at fixed sensors, plus operating parameters | Learned trunk at query coordinates | Function-input learning within the sampled family |

A sampled function from a two-parameter geometry family still does **not** establish generalization to arbitrary geometries. POD modes are a fixed trunk; they must not be described as a trained coordinate network.

**Sensor versus query:** branch sensor positions/order are fixed in the conventional fixed-sensor formulation. Geometry values at those sensors vary by case. Query points may vary by case, but each target must retain its own coordinates. For a cavity, both x and y must match, including meshgrid indexing and flattening order.

**Splits and fitting:** split by physical case before expanding into spatial rows. Fit scalers, POD means and bases on selection-training cases only during model selection. After selection, a declared refit may include validation cases; the public held-out regression set stays excluded from fitting. Early stopping uses validation. A different RNG seed alone does not prove disjoint physical cases. Once test results guide repairs or tuning, reserve fresh cases for the final untouched assessment.

The next cell audits the actual development cavity arrays before POD fitting and deliberately permutes coordinates to check that the guard detects an error. It checks metadata correspondence, not independent solver accuracy. A source export that already mislabeled its coordinates requires an independent check.

In [ ]:
from deeponet_data_audit import audit_pairs, audit_fixed_sensors, audit_case_splits

# Source coordinates use field[y, x] and NumPy C-order flattening.
x_source, y_source = np.asarray(data["x"]), np.asarray(data["y"])
source_xy = np.stack(np.meshgrid(x_source, y_source, indexing="xy"), axis=-1)
source_xy = source_xy.reshape(-1, 2)
dev_ids = np.where(data["split"] == "train")[0]
source_case_ids = np.repeat(data["Re"][dev_ids], len(source_xy))
source_coordinates = np.tile(source_xy, (len(dev_ids), 1))
# Preserve these keys alongside target values; do not shuffle one without the others.
velocity_targets = np.stack((data["u"][dev_ids], data["v"][dev_ids]), axis=-1)
assert velocity_targets.shape == (len(dev_ids), len(y_source), len(x_source), 2)
assert velocity_targets.reshape(-1, 2).shape[0] == len(source_case_ids)
query_coordinates = np.concatenate([source_xy.copy() for _ in dev_ids])
branch_case_ids = np.concatenate([np.full(len(source_xy), re) for re in data["Re"][dev_ids]])
print(audit_pairs(branch_case_ids, source_case_ids, query_coordinates, source_coordinates))
audit_case_splits([100, 150, 200, 250, 300, 350, 400], [225], [175, 275, 375])

bad_queries = query_coordinates.copy()
bad_queries[[0, 1]] = bad_queries[[1, 0]]
try:
    audit_pairs(branch_case_ids, source_case_ids, bad_queries, source_coordinates)
except ValueError as error:
    print("Expected negative control:", error)
else:
    raise AssertionError("The coordinate audit failed to detect the permutation")


## 1. Start from physical validation, not from the network

The labels come from the educational streamfunction–vorticity solver. Its velocity centerlines at $Re=100$ and 400 were compared with Ghia, Ghia, and Shin (1982). The neural operator is useful only if it retains that benchmark fidelity.

Pressure is now a direct output of a separate POD--DeepONet head. Its labels were recovered from the steady momentum-gradient field in Lab 1 and use a zero-mean gauge. Accurate pressure prediction therefore measures fidelity to those labels; it does not independently validate their generation procedure.

In [ ]:
def centerlines(u, v):
    mid = len(data["x"]) // 2
    return u[:, mid], v[mid, :]

idx400 = poddon.case_index(data, 400)
u400, v400 = data["u"][idx400], data["v"][idx400]
uc, vc = centerlines(u400, v400)
ghia = w4utils.GHIA[400]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(uc, data["y"], label="course CFD")
axes[0].scatter(ghia["u"], ghia["y"], label="Ghia et al.")
axes[0].set(xlabel="u/U_lid", ylabel="y/L", title="Vertical centerline")
axes[1].plot(data["x"], vc, label="course CFD")
axes[1].scatter(ghia["x"], ghia["v"], label="Ghia et al.")
axes[1].set(xlabel="x/L", ylabel="v/U_lid", title="Horizontal centerline")
for ax in axes: ax.grid(alpha=.25); ax.legend()
plt.tight_layout()


### Prediction prompt

Before computing the POD, answer:

1. Will the first mode describe the mean primary vortex, a Reynolds-dependent correction, or a wall discontinuity?
2. Which public held-out regression case should be hardest: 175, 275, or 375? Why?
3. Why can a linear combination of divergence-free CFD snapshots preserve divergence more naturally than an unconstrained coordinate MLP?

## 2. Construct a development-only spatial trunk

Concatenate each development velocity case as $[u(:),v(:)]$ and form pressure states separately as $p(:)-\langle p\rangle$. Subtract each development mean and compute two SVDs. Separate trunks prevent pressure scaling from changing the divergence-free velocity modes. No held-out snapshot may enter either mean or SVD.

In [ ]:
development = np.where(data["split"] == "train")[0]
states = poddon.state_matrix(data, development, "velocity")
pressure_states = poddon.state_matrix(data, development, "pressure")
state_mean = states.mean(axis=0)
_, singular_values, spatial_modes = np.linalg.svd(states - state_mean, full_matrices=False)
energy = np.cumsum(singular_values**2) / np.sum(singular_values**2)

display(pd.DataFrame({
    "mode": np.arange(1, len(singular_values)+1),
    "singular_value": singular_values,
    "cumulative_energy": energy,
}))

plt.semilogy(np.arange(1, len(singular_values)+1), singular_values, "o-")
plt.xlabel("mode"); plt.ylabel("singular value"); plt.grid(alpha=.3)


Energy is a representation diagnostic, not the final selection rule. A low-energy mode can still affect a wall layer or weak secondary vortex. Therefore rank is selected by complete-case validation, not by choosing a visually attractive cumulative-energy threshold after seeing the test fields.


## 3. Branch network and leakage-free selection

For a fixed rank, the coefficients are exact projections onto the development trunk. A small tanh MLP maps standardized Reynolds number to standardized coefficients. L-BFGS is appropriate here because only seven physical cases are used during architecture selection.

For each candidate, train seeds 690–692 on the selection-training cases and compute the mean error on the complete $Re=225$ field. This uses Reynolds cases—not random grid points—as the statistical units.

In [ ]:
RUN_SELECTION = False  # Set True to regenerate both head-selection tables.

if RUN_SELECTION:
    selected, selection = poddon.select_model(data)
else:
    selection = pd.read_csv(RESULTS / "deeponet_selection.csv")
    selected = {}
    for head in ("velocity", "pressure"):
        best = selection[selection["head"] == head].sort_values(
            "mean_validation_relative_L2"
        ).iloc[0]
        selected[head] = {
            "rank": int(best["rank"]),
            "hidden": tuple(int(v) for v in best["hidden"].split("x")),
            "input_transform": best["input_transform"],
        }

display(selection[[
    "head", "rank", "hidden", "input_transform", "trunk_energy_fraction",
    "mean_validation_relative_L2",
    "min_validation_relative_L2",
    "max_validation_relative_L2",
]])
print("selected heads:", selected)


**Retained author result, not recomputed by this Markdown cell.**

### Frozen selection result

The development-only rule selected two rank-3 trunks: a `(32,32)` velocity branch using $Re$, and an 8-neuron pressure branch using $\log Re$:

- mean validation velocity error: **0.240%**;
- mean validation pressure error: **0.185%**;
- final velocity/pressure trunk energies after refitting: **99.988%/99.997%**.

In the retained author protocol, the cases then labelled "blind" were excluded from this architecture selection. They are now public held-out regression cases for the student.

## 4. Refit after selection

Now rebuild the trunk using all allowed development cases and fit the selected branch architecture with all three predeclared seeds. The output transform imposes the known no-slip and moving-lid velocity values exactly. This transform uses prescribed boundary conditions, not held-out CFD labels.

In [ ]:
EXPERIMENT_MODE = "retained_demo"  # or "fresh_experiment"
assert EXPERIMENT_MODE in {"retained_demo", "fresh_experiment"}
RUN_REFIT = EXPERIMENT_MODE == "fresh_experiment"
# Every fresh evaluation below consumes the bundles created here.

if RUN_REFIT:
    trunks = {
        head: poddon.make_trunk(data, development, config["rank"], head)
        for head, config in selected.items()
    }
    bundles = [
        poddon.fit_operator(data, development, trunks, selected, seed)
        for seed in poddon.SEEDS
    ]
    print("refit complete")
else:
    print("Using the verified result package. Set EXPERIMENT_MODE to fresh_experiment and rerun this cell for an in-memory refit.")

# Stop: public held-out regression assessment gate

Before running the next cell, write down:

- selected rank and branch widths;
- development and public held-out Reynolds lists, including prior exposure;
- seeds;
- primary and physical metrics; and
- the failure criterion you would report.

Do not treat this public assessment as new blind evidence. If these metrics inform model selection, record that development use; a new blind assessment requires new instructor-withheld cases.

In [ ]:
if RUN_REFIT:
    from datetime import datetime, timezone
    import uuid
    STUDENT_OUTPUT = ROOT / "student_outputs" / ("pod_deeponet_" + uuid.uuid4().hex)
    metrics, fresh_predictions, fresh_protocol = poddon.evaluate_fresh_operator(
        data, bundles, selected, STUDENT_OUTPUT)
    blind = metrics.copy()
    print("FRESH experiment", fresh_protocol["run_id"], "saved to", STUDENT_OUTPUT)
else:
    metrics = pd.read_csv(RESULTS / "deeponet_metrics.csv")
    blind = metrics[metrics["method"] == "three-seed POD-DeepONet ensemble"].copy()
    print("RETAINED AUTHOR EVIDENCE: these metrics do not evaluate a new fit.")
blind["E_uv_percent"] = 100 * blind["relative_L2_uv"]
blind["E_p_percent"] = 100 * blind["relative_L2_p"]
display(blind[["Re", "E_uv_percent", "E_p_percent", "div_l2_pred", "wall_rms_error"]])
# Report failure rather than repeatedly tuning the public held-out cases.
checks = {"wall": bool(blind["wall_rms_error"].max() < 1e-12),
          "divergence": bool(blind["div_l2_pred"].max() < 1e-12),
          "velocity": bool(blind["relative_L2_uv"].max() < .005),
          "pressure": bool(blind["relative_L2_p"].max() < .005)}
print("Predeclared demonstration criteria:", checks)

**Retained author result, not recomputed by this Markdown cell.**

### Historical author "blind" result—every now-public case reported

| held-out $Re$ | ensemble $E_{uv}$ | ensemble $E_p$ | divergence $L_2$ |
|---:|---:|---:|---:|
| 175 | **0.4455%** | **0.2506%** | $1.31\times10^{-15}$ |
| 275 | **0.0727%** | **0.1245%** | $1.33\times10^{-15}$ |
| 375 | **0.0947%** | **0.1073%** | $1.42\times10^{-15}$ |

This is not a best-seed table. The pointwise ensemble and the full seed range are both retained.

## 5. Field evidence and its provenance
The next code cell plots the current in-memory ensemble in `fresh_experiment` mode.
The linked figure below is **historical author evidence only**, not the result of any new refit.

![Retained author POD-DeepONet result](../../results/pod_deeponet/pod_deeponet_ghia_validation.svg)

In [ ]:
if RUN_REFIT:
    re_plot = float(blind["Re"].iloc[0])
    idx = poddon.case_index(data, re_plot)
    up, vp, pp = fresh_predictions[re_plot]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
    for ax, values, title in zip(axes, [np.hypot(data["u"][idx], data["v"][idx]),
                np.hypot(up, vp), np.hypot(up-data["u"][idx], vp-data["v"][idx])],
                ["CFD label", "Fresh ensemble speed", "Fresh vector error"]):
        im=ax.contourf(data["x"], data["y"], values, 25); fig.colorbar(im, ax=ax); ax.set_title(title)
    fig.suptitle(f"Fresh run {fresh_protocol['run_id']} — Re={re_plot:g}")
    fig.tight_layout(); fig.savefig(STUDENT_OUTPUT / "field_evidence.png"); plt.show()

## 6. What is the neural-network advantage?

The claim is not that a surrogate improves the Ghia data. The reference CFD solve establishes the physics label; the network should retain that fidelity while avoiding a new iterative PDE solve for each query.

The reported timing includes reconstruction of all three $65\times65$ fields from a three-seed ensemble. The CFD timing is an independently rerun streamfunction–vorticity solve including pressure recovery, stopped at residual $8.95\times10^{-7}$.

In [ ]:
if RUN_REFIT:
    display(blind[["Re", "prediction_seconds"]])
    fresh_ghia = []
    for re_value in (100, 400):
        prediction = poddon.ensemble_predict(bundles, re_value, data["u"].shape[1:])
        up, vp, _ = prediction
        ghia = w4utils.GHIA[re_value]
        mid = len(data["x"]) // 2
        uc = np.interp(ghia["y"], data["y"], up[:, mid])
        vc = np.interp(ghia["x"], data["x"], vp[mid, :])
        fresh_ghia.append({"Re": re_value, "u_Ghia_RMSE": float(np.sqrt(np.mean((uc-np.asarray(ghia["u"]))**2))),
                           "v_Ghia_RMSE": float(np.sqrt(np.mean((vc-np.asarray(ghia["v"]))**2)))})
    display(pd.DataFrame(fresh_ghia))
    pd.DataFrame(fresh_ghia).to_csv(STUDENT_OUTPUT / "ghia_metrics.csv", index=False)
    print("No new CFD solve was timed; no fresh speedup ratio is claimed.")
else:
    display(pd.read_csv(RESULTS / "deeponet_ghia_metrics.csv"))
    timing = json.loads((RESULTS / "deeponet_protocol_and_timing.json").read_text())
    display(pd.DataFrame([timing]))
    print("Retained timing on the author's recorded environment; not this session's timing.")

**Retained author result, not recomputed by this Markdown cell.**

### Ghia and timing result

| $Re$ | CFD $E_u$ | POD-DeepONet $E_u$ | CFD $E_v$ | POD-DeepONet $E_v$ |
|---:|---:|---:|---:|---:|
| 100 | 0.008594 | 0.008666 | 0.024205 | 0.024363 |
| 400 | 0.052393 | 0.052474 | 0.154259 | 0.154479 |

The neural field preserves the benchmark fidelity of the educational CFD labels. In the recorded CPU run, three-field ensemble inference was about **1.0 ms** and the CFD solve about **5.3 s**, an amortized per-case advantage near **$5.3\times10^3\times$**. Report timing with hardware/software context and never include training cost in a per-query speedup without stating how many queries amortize it.

## 7. Interpretation and limitations

1. **Why divergence is near round-off:** the fixed trunk is a linear basis of divergence-free CFD snapshots; the output transform also restores exact wall values.
2. **Why the first public held-out regression case is harder:** $Re=175$ lies in a part of coefficient space where the branch mapping is more sensitive to sparse parameter sampling.
3. **Why this is still restricted:** a scalar Reynolds input does not describe a new lid-velocity function, geometry, or boundary shape.
4. **Why Ghia errors do not vanish:** the neural operator reproduces the educational CFD solver, including its grid/discretization error. Label agreement alone cannot establish greater physical accuracy. Denoising or regularization might improve a particular physical error, but that requires independent evidence.
5. **What pressure accuracy means:** the pressure head predicts stored zero-mean labels directly, but it inherits their momentum-recovery and discretization limitations.
6. **Why speedup is useful:** parametric studies, uncertainty propagation, optimization, and interactive teaching require many field queries after training.

## Common failure modes and fixes

| Symptom | Likely cause | Evidence-based fix |
|---|---|---|
| Very small training error, poor public held-out regression fields | coefficient overfit from only a few cases | reduce rank/branch width; add complete development cases |
| Nonzero wall error | boundary transform omitted or corner convention changed | inspect all four walls and preserve the CFD corner convention |
| Large divergence | predicted $u,v$ directly with an unconstrained trunk | use a streamfunction/POD trunk or add a carefully scaled divergence loss |
| Excellent global error, wrong centerline | global norm hides a local feature | retain Ghia and vortex/centerline diagnostics |
| Unrealistic speedup | data loading or solver convergence excluded inconsistently | time matched end-to-end calls; report warm-up and repetitions |
| Better test result after several redesigns | test set became a validation set | create a new untouched physical test family |

## Exercises

1. Plot the first three shared trunk modes and identify the flow structure carried by each.
2. Repeat selection with rank 2 included. Does validation—not energy alone—reject it?
3. Compare one seed, the three-seed ensemble, and the seed spread at every public held-out regression case.
4. Remove the boundary output transform and quantify wall, global, and divergence changes.
5. Build a true operator dataset in which the branch input is $[U_{lid}(\xi_1),\ldots,U_{lid}(\xi_m)]$, not a scalar Reynolds number.
6. Repeat the timing for 1, 10, 100, and 1000 queries. At what query count does offline work amortize?
7. Compare the direct pressure head with pressure reconstructed from predicted velocity, then explain why the two errors answer different questions.

## Required deliverables

- selection table with complete case lists and three seeds;
- held-out table with every Reynolds case, velocity/pressure errors, seed range, wall error, divergence, and pressure gauge;
- one field/error figure and both centerline comparisons;
- Ghia table for $Re=100$ and 400;
- matched timing table with residual and software/hardware context;
- a paragraph stating the restricted scope of scalar-branch POD-DeepONet; and
- one failure mode or limitation that remains after the final model.

### Suggested report claim

> After development-only rank/architecture selection, the multi-output POD-DeepONet preserved the Ghia fidelity of the reference solver, maintained round-off-level divergence, achieved sub-0.5% velocity and pressure errors on all retained test fields, and reduced amortized three-field query time from seconds to milliseconds. The result applies to the fixed-geometry, scalar-Reynolds family and is not evidence of arbitrary boundary-function generalization.

## References and next steps

- Ghia, Ghia & Shin (1982), *Journal of Computational Physics* 48, 387–411.
- Lu et al. (2021), “Learning nonlinear operators via DeepONet,” *Nature Machine Intelligence* 3, 218–229.
- Berkooz, Holmes & Lumley (1993), POD in turbulent flows.
- Hesthaven & Ubbiali (2018), non-intrusive neural reduced-order modeling.

Continue with the Week-5 POD, physics-guided-loss, or uncertainty track. For a research extension, replace the scalar branch with a sampled boundary function or a geometry encoding and create new complete blind operators.
